In [7]:
import cv2
from doclayout_yolo import YOLOv10

# Load the pre-trained model
model = YOLOv10("test/doclayout_yolo_docstructbench_imgsz1024.pt")

# Perform prediction
det_res = model.predict("/home/bas/Documents/Visual Code Data/Natuurtijdschriften/PAGE-0001-003.tif",   # Image to predict
    imgsz=2048,        # Prediction image size
    conf=0.2,          # Confidence threshold
    device="cuda:0"    # Device to use (e.g., 'cuda:0' or 'cpu')
)

# Annotate and save the result
annotated_frame = det_res[0].plot(pil=True, line_width=5, font_size=20)
cv2.imwrite("result_docyolo.jpg", annotated_frame)


image 1/1 /home/bas/Documents/Visual Code Data/Natuurtijdschriften/PAGE-0001-003.tif: 2048x1472 2 titles, 14 plain texts, 3 abandons, 1 figure, 36.9ms
Speed: 8.2ms preprocess, 36.9ms inference, 0.5ms postprocess per image at shape (1, 3, 2048, 1472)


True

# fijn tuunen

In [13]:
import json
import os

def convert_coco_to_yolo(coco_file, output_dir):
    # Load COCO-style JSON file
    with open(coco_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Build a mapping from image_id → image_info for quick lookup
    image_id_to_info = {img["id"]: img for img in data["images"]}

    # Build a mapping from COCO category_id → YOLO class index (0-based)
    sorted_categories = sorted(data["categories"], key=lambda x: x["id"])
    category_id_to_index = {cat["id"]: idx for idx, cat in enumerate(sorted_categories)}

    # Group annotations by image_id
    annotations_by_image = {}
    for ann in data["annotations"]:
        annotations_by_image.setdefault(ann["image_id"], []).append(ann)

    # Convert each image's annotations to YOLO format
    for image_id, image_info in image_id_to_info.items():
        width = image_info["width"]
        height = image_info["height"]

        # Build YOLO label filename based on image basename
        img_basename = os.path.basename(image_info["file_name"])
        label_filename = os.path.splitext(img_basename)[0] + ".txt"
        label_path = os.path.join(output_dir, label_filename)

        with open(label_path, "w", encoding="utf-8") as f:
            for ann in annotations_by_image.get(image_id, []):
                category_id = ann["category_id"]
                if category_id not in category_id_to_index:
                    continue  # skip unknown categories

                yolo_class = category_id_to_index[category_id]
                bbox = ann["bbox"]  # [x_min, y_min, width, height]

                # Convert COCO bbox → YOLO bbox [x_center, y_center, width, height], normalized
                x_center = (bbox[0] + bbox[2] / 2) / width
                y_center = (bbox[1] + bbox[3] / 2) / height
                norm_width = bbox[2] / width
                norm_height = bbox[3] / height

                f.write(f"{yolo_class} {x_center:.6f} {y_center:.6f} {norm_width:.6f} {norm_height:.6f}\n")

    print(f"Conversion complete! YOLO labels saved to: {output_dir}")

# Example usage
convert_coco_to_yolo("test/fine_tune_Yolo/train/val.json", "test/fine_tune_Yolo/")


Conversion complete! YOLO labels saved to: test/fine_tune_Yolo/


# trènen

## Train in een aparte omgeving